In [ ]:
import sys

sys.path.append("..")

In [ ]:
import numpy as np
import pandas as pd
import torch
from sklearn import metrics
from sklearn.model_selection import StratifiedKFold, train_test_split
from torch.utils.data import DataLoader, TensorDataset


In [ ]:
from src.data import load_data, preprocess_data

In [ ]:
df = load_data()
df = preprocess_data(df)

In [ ]:
df

In [ ]:
type(df['one_hot_sequence'][0])

for pytorch we need this in the form of 4 * 57 and not 57 * 4. Hence we will transpose it

In [ ]:
df['one_hot_sequence'] = df['one_hot_sequence'].apply(lambda x: x.T)

In [ ]:
df['one_hot_sequence'][0].shape

In [ ]:
df

Let us split this into test and training data

In [ ]:
X = df['one_hot_sequence']
y = df['class']

In [ ]:
X.shape

In [ ]:
X[0].shape

Let us stack all the

In [ ]:
X = np.stack(X.values)
X.shape

In [ ]:
type(X)

In [ ]:
y.shape

In [ ]:
y = y.to_numpy()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=13, shuffle=True)

In [ ]:
type(X_train[0])

let us define the cnn

In [ ]:
class CNN(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = torch.nn.Conv1d(in_channels=4, out_channels=16, kernel_size=4)
        self.conv2 = torch.nn.Conv1d(in_channels=16, out_channels=32, kernel_size=4)
        self.pool1 = torch.nn.MaxPool1d(kernel_size=2)
        self.pool2 = torch.nn.MaxPool1d(kernel_size=2)
        self.relu = torch.nn.ReLU()
        self.fc1 = torch.nn.Linear(in_features=384, out_features=10)
        self.fc2 = torch.nn.Linear(in_features=10, out_features=1)
        
    def forward(self, x):
        x = self.conv1(x)
        x = self.relu(x)
        x = self.pool1(x)
        x = self.conv2(x)
        x = self.relu(x)
        x = self.pool2(x)
        x = torch.flatten(x, 1)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x

In [ ]:
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

In [ ]:
torch.version.cuda

In [ ]:
model = CNN().to(device)
model

In [ ]:
skf = StratifiedKFold(n_splits=4, shuffle=True, random_state=13)
splits = list(skf.split(X_train, y_train))

In [ ]:
print(type(X_train))
print(type(y_train))
print(type(X_test))
print(type(y_test))

In [ ]:
cnn_metrics = {
    'accuracy': [],
    'recall': [],
    'precision': [],
    'f1_score': [],
    'auc': []
}
cnn_cm = np.zeros((2,2), dtype=int)

In [ ]:
for fold, (train_idx, val_idx) in enumerate(splits):
    
    # splitting the data into training and validation sets for this fold
    X_train_fold, X_val_fold = X_train[train_idx], X_train[val_idx]
    y_train_fold, y_val_fold = y_train[train_idx], y_train[val_idx]
    
    # converting it to tensors because PyTorch models work with tensors
    X_train_tensor = torch.tensor(X_train_fold, dtype=torch.float32)
    X_val_tensor = torch.tensor(X_val_fold, dtype=torch.float32)
    y_train_tensor = torch.tensor(y_train_fold, dtype=torch.float32).unsqueeze(1)  # unsqueeze to add a dimension for the output to match the model's output
    y_val_tensor = torch.tensor(y_val_fold, dtype=torch.float32).unsqueeze(1) # models output is of shape (batch_size, 1) so we need to make sure our target is also of the same shape
    
    # creating datasets and dataloaders for training and validation
    train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
    val_dataset = TensorDataset(X_val_tensor, y_val_tensor)
    
    train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)
    
    # defining the model, loss function, and optimizer
    model = CNN().to(device)
    criterion = torch.nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    
    # training the model for a number of epochs
    epochs = 15
    
    for epoch in range(epochs):
        for data, target in train_loader:
            data = data.to(device)
            target = target.to(device)
            
            model.train()
            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, target)
            loss.backward()
            optimizer.step()
        
    model.eval()
    target_array = []
    pred_array = []
    with torch.no_grad():
        for data, target in val_loader:
            data = data.to(device)
            target = target.to(device)
            
            pred = model(data)
            
            # converting logits to probabilities
            pred = torch.sigmoid(pred)
            
            # converting prbabilities to binary predictions
            pred = (pred > 0.5).float()
            
            # convert target to 1D tensor for compatibility with sklearn metrics
            target = target.squeeze(1)
            
            # converting tensors to numpy arrays for sklearn metrics
            pred_array = np.append(pred_array, pred.cpu().numpy())
            target_array = np.append(target_array, target.cpu().numpy())
            
            # updating the confusion matrix
        cnn_cm += metrics.confusion_matrix(target_array, pred_array, labels=[0, 1])
        
        #updating other metrics
        cnn_metrics['accuracy'].append(metrics.accuracy_score(target_array, pred_array))
        cnn_metrics['recall'].append(metrics.recall_score(target_array, pred_array))
        cnn_metrics['precision'].append(metrics.precision_score(target_array, pred_array))
        cnn_metrics['f1_score'].append(metrics.f1_score(target_array, pred_array))
        
        cnn_metrics['auc'].append(metrics.roc_auc_score(target_array, pred_array))


In [ ]:
cnn_metrics

In [ ]:
metrics_df = pd.DataFrame(cnn_metrics)
metrics_df

In [ ]:
metrics_df

In [ ]:
metrics_df = pd.DataFrame(index=cnn_metrics.keys(), columns=['cv metric'])
for metric, values in cnn_metrics.items():
    metrics_df.loc[metric] = f'{np.mean(values):.3f} ± {np.std(values):.3f}'

In [ ]:
cnn_cm